In [1]:
# -------------------------------------------------------------------------
# 0. Install ALL torch packages together to prevent CPU override
# -------------------------------------------------------------------------
!pip install torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121 -q
!pip install torchxrayvision --no-deps -q
!pip install scikit-image -q

import torch
assert torch.cuda.is_available(), "CUDA not available!"
assert torch.version.cuda is not None, "CPU-only torch loaded!"
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")

_t = torch.randn(2, 2, device='cuda') @ torch.randn(2, 2, device='cuda')
print(f"CUDA test: PASSED")
del _t

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 101.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 103.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 207.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 158.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 63.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 128.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 185.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 146.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 106.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import sys
import json
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

EXPECTED_TOTAL_IMAGES_P10_P11 = 75216

# -------------------------------------------------------------------------
# 1. Fetch Official Metadata and Labels from MIMIC-CXR-JPG
# -------------------------------------------------------------------------
print("[Phase 1/5] Fetching metadata and CheXpert labels from MIMIC-CXR-JPG...")

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    PHYSIONET_USER = user_secrets.get_secret("PHYSIONET_USER")
    PHYSIONET_PASS = user_secrets.get_secret("PHYSIONET_PASS")
except Exception:
    PHYSIONET_USER = os.environ.get("PHYSIONET_USER", "")
    PHYSIONET_PASS = os.environ.get("PHYSIONET_PASS", "")

netrc_path = os.path.expanduser("~/.netrc")
with open(netrc_path, "w", encoding="utf-8") as f:
    f.write(f"machine physionet.org\nlogin {PHYSIONET_USER}\npassword {PHYSIONET_PASS}\n")
os.chmod(netrc_path, 0o600)

metadata_files = [
    ("meta.csv", "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv.gz"),
    ("chexpert.csv", "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-chexpert.csv.gz"),
]

for target_name, url in metadata_files:
    if not os.path.exists(target_name):
        gz_name = f"{target_name}.gz"
        subprocess.run(f"wget -q --netrc -O {gz_name} {url}", shell=True, check=True)
        subprocess.run(f"gunzip -f {gz_name}", shell=True, check=True)

print("Metadata and labels ready.")

# -------------------------------------------------------------------------
# 2. Build Unified File Index Across Attached Datasets
# -------------------------------------------------------------------------
print("[Phase 2/5] Building unified index across all attached input datasets...")

input_base = Path("/kaggle/input")
file_index = {}
impression_file_path = None

for root, _, files in os.walk(input_base):
    for f in files:
        full_path = os.path.join(root, f)
        # Identify uploaded impression CSV
        if f in ["mimic_train_impressions.csv", "train_imp.csv"]:
            impression_file_path = full_path
        # Index JPG images
        elif f.endswith(".jpg"):
            parts = full_path.split("/")
            rel_path = None
            for prefix in ("p10", "p11"):
                if prefix in parts:
                    idx = parts.index(prefix)
                    rel_path = "/".join(parts[idx:])
                    break
            if rel_path is not None and rel_path not in file_index:
                file_index[rel_path] = full_path

n_indexed = len(file_index)
print(f"Total unique images indexed: {n_indexed:,}")

coverage_pct = (n_indexed / EXPECTED_TOTAL_IMAGES_P10_P11) * 100
print(f"Dataset pool coverage relative to theoretical target ({EXPECTED_TOTAL_IMAGES_P10_P11:,}): {coverage_pct:.1f}%")

with open(f"{work_dir}/unified_file_index.json", "w", encoding="utf-8") as f:
    json.dump(file_index, f)

# -------------------------------------------------------------------------
# 3. Merge Metadata, CheXpert Labels, and Report Impressions
# -------------------------------------------------------------------------
print("[Phase 3/5] Merging metadata, clinical labels, and report text...")

if not impression_file_path:
    sys.exit("Error: mimic_train_impressions.csv not found in /kaggle/input/. Please upload the CSV via '+ Add Input'.")

print(f"Loading impression text from: {impression_file_path}")
report_df = pd.read_csv(impression_file_path)

meta_df = pd.read_csv("meta.csv")
chex_df = pd.read_csv("chexpert.csv")

# Filter metadata strictly for p10 and p11 cohorts
mask_cohort = (meta_df["subject_id"] >= 10000000) & (meta_df["subject_id"] < 12000000)
meta_cohort = meta_df[mask_cohort].copy()

label_cols = [c for c in chex_df.columns if c not in ["subject_id", "study_id"]]
merged_df = pd.merge(meta_cohort, chex_df[["study_id"] + label_cols], on="study_id", how="inner")

# Merge report impressions by dicom_id
merged_df = pd.merge(
    merged_df,
    report_df[["dicom_id", "report"]],
    on="dicom_id",
    how="left"
)

n_before_filter = len(merged_df)
n_null_reports = merged_df["report"].isna().sum()
print(f"Null reports detected: {n_null_reports:,} out of {n_before_filter:,} ({n_null_reports / n_before_filter * 100:.1f}%)")

merged_df = merged_df.dropna(subset=["report"]).reset_index(drop=True)
print(f"Retained records after removing empty reports: {len(merged_df):,}")

# Build local disk path with explicit integer casting
def get_path(row):
    subject_id = int(row["subject_id"])
    study_id = int(row["study_id"])
    prefix = "p10" if subject_id < 11000000 else "p11"
    rel = f"{prefix}/p{subject_id}/s{study_id}/{row['dicom_id']}.jpg"
    return file_index.get(rel, None)

merged_df["image_path"] = merged_df.apply(get_path, axis=1)

initial_count = len(merged_df)
merged_df = merged_df.dropna(subset=["image_path"]).reset_index(drop=True)
print(f"Matched {len(merged_df):,} valid images on disk with clinical report text.")

if len(merged_df) == 0:
    sys.exit("Zero matching records found. Verify directory index.")

# -------------------------------------------------------------------------
# 4. Strict Patient-Level Split (80% Train / 10% Val / 10% Test)
# -------------------------------------------------------------------------
print("[Phase 4/5] Executing strict patient-level split (0% leakage)...")

np.random.seed(42)
gss_train = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(gss_train.split(merged_df, groups=merged_df["subject_id"]))

train_df = merged_df.iloc[train_idx].copy()
temp_df = merged_df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df["subject_id"]))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

master_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

train_pts = set(train_df["subject_id"])
val_pts = set(val_df["subject_id"])
test_pts = set(test_df["subject_id"])

print(f"\nSplit Distribution:")
print(f"  Train : {len(train_df):,} images | {train_df['study_id'].nunique():,} studies | {len(train_pts):,} patients")
print(f"  Val   : {len(val_df):,} images | {val_df['study_id'].nunique():,} studies | {len(val_pts):,} patients")
print(f"  Test  : {len(test_df):,} images | {test_df['study_id'].nunique():,} studies | {len(test_pts):,} patients")

assert len(train_pts & val_pts) == 0 and len(train_pts & test_pts) == 0 and len(val_pts & test_pts) == 0, \
    "Patient leakage detected between splits. Halting."

print("Patient Leakage Audit: 0 patients shared across splits (Passed).")

# -------------------------------------------------------------------------
# 5. Pathology Audit and Export Final Manifest
# -------------------------------------------------------------------------
print("\n[Phase 5/5] Pathology Label Distribution Audit:")

print(f"{'Pathology':<30} | {'Test Pos':<10} | {'Val Pos':<10} | {'Train Pos':<10}")
print("-" * 75)
for col in label_cols:
    t_pos = int((test_df[col] == 1.0).sum())
    v_pos = int((val_df[col] == 1.0).sum())
    tr_pos = int((train_df[col] == 1.0).sum())
    print(f"{col:<30} | {t_pos:<10,} | {v_pos:<10,} | {tr_pos:<10,}")

output_csv = f"{work_dir}/mimic_master_split.csv"
master_df.to_csv(output_csv, index=False)

status_report = {
    "total_indexed_images": n_indexed,
    "expected_total_images": EXPECTED_TOTAL_IMAGES_P10_P11,
    "coverage_pct": round(coverage_pct, 2),
    "final_manifest_rows": len(master_df),
    "total_unique_studies": int(master_df["study_id"].nunique()),
    "total_unique_patients": int(master_df["subject_id"].nunique()),
}

with open(f"{work_dir}/pipeline_status_report.json", "w", encoding="utf-8") as f:
    json.dump(status_report, f, indent=2)

print(f"\nMaster manifest saved successfully to: {output_csv}")
print(f"Pipeline status report saved to: {work_dir}/pipeline_status_report.json")

[Phase 1/5] Fetching metadata and CheXpert labels from MIMIC-CXR-JPG...
Metadata and labels ready.
[Phase 2/5] Building unified index across all attached input datasets...
Total unique images indexed: 73,995
Dataset pool coverage relative to theoretical target (75,216): 98.4%
[Phase 3/5] Merging metadata, clinical labels, and report text...
Loading impression text from: /kaggle/input/datasets/alinikoukar/mimic-train-impressions/mimic_train_impressions.csv
Null reports detected: 11,707 out of 75,215 (15.6%)
Retained records after removing empty reports: 63,508
Matched 62,450 valid images on disk with clinical report text.
[Phase 4/5] Executing strict patient-level split (0% leakage)...

Split Distribution:
  Train : 50,616 images | 29,900 studies | 9,704 patients
  Val   : 6,103 images | 3,521 studies | 1,213 patients
  Test  : 5,731 images | 3,320 studies | 1,213 patients
Patient Leakage Audit: 0 patients shared across splits (Passed).

[Phase 5/5] Pathology Label Distribution Audit:
P

In [3]:
# -------------------------------------------------------------------------
# 0. Safe Dependency Installation (DO NOT touch PyTorch version)
# -------------------------------------------------------------------------
!pip install torchxrayvision --no-deps -q
!pip install scikit-image -q

import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm import tqdm
import torchxrayvision as xrv

# -------------------------------------------------------------------------
# 1. CUDA Diagnostics & Hardware Verification
# -------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if device.type == "cuda":
    print(f"  PyTorch version  : {torch.__version__}")
    print(f"  CUDA version     : {torch.version.cuda}")
    print(f"  GPU              : {torch.cuda.get_device_name(0)}")
    print(f"  Compute Capability: {torch.cuda.get_device_capability(0)}")
    # Quick sanity: basic CUDA tensor op
    _test = torch.randn(2, 2, device=device) @ torch.randn(2, 2, device=device)
    print(f"  CUDA tensor test : PASSED (shape={_test.shape})")
    del _test

# -------------------------------------------------------------------------
# 2. Load Manifest and Tokenizer
# -------------------------------------------------------------------------
manifest_path = "/kaggle/working/mimic_master_split.csv"
master_df = pd.read_csv(manifest_path)

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
special_tokens = {"bos_token": "<bos>", "eos_token": "<eos>", "pad_token": "<pad>"}
tokenizer.add_special_tokens(special_tokens)
vocab_size = len(tokenizer)
pad_id = tokenizer.pad_token_id
bos_id = tokenizer.bos_token_id
eos_id = tokenizer.eos_token_id

# -------------------------------------------------------------------------
# 3. XRV Feature Dimensions Sanity Check (on CPU to avoid issues)
# -------------------------------------------------------------------------
print("\n[Sanity Check] Verifying TorchXRayVision DenseNet121 feature dimensions...")
temp_densenet = xrv.models.DenseNet(weights="densenet121-res224-mimic_ch")
dummy_input = torch.randn(2, 1, 224, 224)
with torch.no_grad():
    dummy_out = temp_densenet.features(dummy_input)
print(f"  Input shape  : {dummy_input.shape}")
print(f"  Output shape : {dummy_out.shape} (Expected: [2, 1024, 7, 7])")
assert dummy_out.shape == (2, 1024, 7, 7), "Dimension mismatch!"
print("  XRV Encoder verified successfully.\n")
del temp_densenet, dummy_input, dummy_out

# -------------------------------------------------------------------------
# 4. Dataset and DataLoaders
# -------------------------------------------------------------------------
class BaselineDataset(Dataset):
    def __init__(self, df, tokenizer, max_seq_len=100):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        try:
            img = Image.open(row['image_path']).convert('L').resize((224, 224))
            img_np = np.array(img, dtype=np.float32)
            img_normalized = xrv.datasets.normalize(img_np, maxval=255)
            image_tensor = torch.tensor(img_normalized, dtype=torch.float32).unsqueeze(0)
        except Exception:
            image_tensor = torch.zeros((1, 224, 224), dtype=torch.float32)

        report_text = str(row['report']) if pd.notna(row['report']) else ""
        tokenized = self.tokenizer(
            f"<bos> {report_text} <eos>",
            max_length=self.max_seq_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "image": image_tensor,
            "input_ids": tokenized["input_ids"].squeeze(0)
        }

train_df = master_df[master_df['split'] == 'train']
val_df = master_df[master_df['split'] == 'val']
test_df = master_df[master_df['split'] == 'test']

train_dataset = BaselineDataset(train_df, tokenizer, max_seq_len=100)
val_dataset = BaselineDataset(val_df, tokenizer, max_seq_len=100)
test_dataset = BaselineDataset(test_df, tokenizer, max_seq_len=100)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders Prepared:")
print(f"  Train : {len(train_loader):,} batches ({len(train_dataset):,} samples)")
print(f"  Val   : {len(val_loader):,} batches ({len(val_dataset):,} samples)")
print(f"  Test  : {len(test_loader):,} batches ({len(test_dataset):,} samples)")

# -------------------------------------------------------------------------
# 5. Architecture: Baseline A (XRV DenseNet121 + LSTM)
# -------------------------------------------------------------------------
class BaselineDenseNetLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, pad_token_id=0):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id
        self.hidden_dim = hidden_dim
        
        # Medical Encoder (Pretrained on MIMIC-CXR)
        densenet = xrv.models.DenseNet(weights="densenet121-res224-mimic_ch")
        self.encoder = densenet.features
        for param in self.encoder.parameters():
            param.requires_grad = False
            
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.init_h = nn.Linear(1024, hidden_dim)
        self.init_c = nn.Linear(1024, hidden_dim)
        
        # Language Decoder
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def extract_visual_features(self, images):
        with torch.no_grad():
            features = self.encoder(images)
        return self.global_pool(features).flatten(1)

    def forward(self, images, input_ids):
        features = self.extract_visual_features(images)
        h_0 = torch.tanh(self.init_h(features)).unsqueeze(0)
        c_0 = torch.tanh(self.init_c(features)).unsqueeze(0)
        embeddings = self.embedding(input_ids[:, :-1])
        lstm_out, _ = self.lstm(embeddings, (h_0, c_0))
        return self.fc_out(lstm_out)

    @torch.no_grad()
    def greedy_generate(self, image, max_len=50):
        self.eval()
        if image.dim() == 3:
            image = image.unsqueeze(0)
        features = self.extract_visual_features(image)
        h = torch.tanh(self.init_h(features)).unsqueeze(0)
        c = torch.tanh(self.init_c(features)).unsqueeze(0)
        
        current_token = torch.tensor([[bos_id]], device=image.device)
        generated = []
        for _ in range(max_len):
            emb = self.embedding(current_token)
            out, (h, c) = self.lstm(emb, (h, c))
            next_id = self.fc_out(out.squeeze(1)).argmax(-1).item()
            if next_id == eos_id:
                break
            generated.append(next_id)
            current_token = torch.tensor([[next_id]], device=image.device)
        return generated

# -------------------------------------------------------------------------
# 6. Initialize Model, Optimizer, Loss
# -------------------------------------------------------------------------
model = BaselineDenseNetLSTM(vocab_size=vocab_size, pad_token_id=pad_id).to(device)

optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=3e-4, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
scaler = torch.amp.GradScaler("cuda")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nBaseline A Initialized. Trainable Parameters: {trainable_params:,}")

# Quick GPU forward pass test before full training
print("[GPU Test] Running single forward pass on GPU...")
_tb = next(iter(train_loader))
_ti, _tt = _tb["image"].to(device), _tb["input_ids"].to(device)
with torch.amp.autocast("cuda"):
    _to = model(_ti, _tt)
print(f"  Forward pass output: {_to.shape} — GPU pipeline PASSED.\n")
del _tb, _ti, _tt, _to
torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 7. Training and Validation Loops
# -------------------------------------------------------------------------
EPOCHS = 10
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for batch in loop:
        images = batch["image"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        targets = input_ids[:, 1:].contiguous()
        
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            logits = model(images, input_ids)
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            targets = input_ids[:, 1:].contiguous()
            with torch.amp.autocast("cuda"):
                logits = model(images, input_ids)
                loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    print(f"\n--- Epoch {epoch}: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f} ---")
    
    # Sample Generation
    sample_img = val_dataset[0]["image"].to(device)
    gen_ids = model.greedy_generate(sample_img, max_len=50)
    print(f"Ground Truth  : {val_df.iloc[0]['report']}")
    print(f"Greedy Output : {tokenizer.decode(gen_ids, skip_special_tokens=True)}")
    print("=" * 75)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "/kaggle/working/baseline_a_best.pt")
        print(">>> Best Checkpoint Saved <<<")

Using compute device: cuda
  PyTorch version  : 2.4.1+cu121
  CUDA version     : 12.1
  GPU              : Tesla P100-PCIE-16GB
  Compute Capability: (6, 0)
  CUDA tensor test : PASSED (shape=torch.Size([2, 2]))


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (17 outputs left out)

Epoch 9/10 [Train]: 100%|██████████| 1582/1582 [31:06<00:00,  1.18s/it, loss=1.1871]



--- Epoch 9: Train Loss = 1.3026 | Val Loss = 1.6291 ---
Ground Truth  : No acute cardiopulmonary process.
Greedy Output :  No acute cardiopulmonary process. 
>>> Best Checkpoint Saved <<<


Epoch 10/10 [Train]: 100%|██████████| 1582/1582 [34:06<00:00,  1.29s/it, loss=1.1841]



--- Epoch 10: Train Loss = 1.2564 | Val Loss = 1.6286 ---
Ground Truth  : No acute cardiopulmonary process.
Greedy Output :  No acute cardiopulmonary process. 
>>> Best Checkpoint Saved <<<


In [4]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation and NLP Resource Downloads
# -------------------------------------------------------------------------
!pip install rouge-score nltk seaborn matplotlib -q

import os
import json
import torch
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, multilabel_confusion_matrix

# Download NLTK tokenization assets
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# -------------------------------------------------------------------------
# 1. Load Best Trained Model Checkpoint
# -------------------------------------------------------------------------
checkpoint_path = "/kaggle/working/baseline_a_best.pt"
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Best model checkpoint not found at: {checkpoint_path}")

print(f"[Evaluation] Loading weights from {checkpoint_path}...")
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

smooth = SmoothingFunction().method1
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# 14 Standard Clinical Pathologies and Matching Lexicons
clinical_keywords = {
    'Atelectasis': ['atelectasis', 'atelectatic', 'collapse', 'volume loss'],
    'Cardiomegaly': ['cardiomegaly', 'enlarged heart', 'enlarged cardiac', 'cardiac enlargement'],
    'Consolidation': ['consolidation', 'airspace disease', 'airspace opacity'],
    'Edema': ['edema', 'vascular congestion', 'fluid overload', 'interstitial edema'],
    'Enlarged Cardiomediastinum': ['widened mediastinum', 'mediastinal widening', 'cardiomediastinum'],
    'Fracture': ['fracture', 'fractured', 'broken rib', 'healing fracture'],
    'Lung Lesion': ['lung lesion', 'nodule', 'mass', 'nodular opacity'],
    'Lung Opacity': ['opacity', 'opacities', 'infiltrate', 'clouding'],
    'No Finding': ['normal', 'clear', 'unremarkable', 'no acute', 'negative'],
    'Pleural Effusion': ['pleural effusion', 'effusion', 'blunting', 'fluid in pleural'],
    'Pleural Other': ['pleural thickening', 'fibrosis', 'calcification'],
    'Pneumonia': ['pneumonia', 'infectious process', 'bronchopneumonia'],
    'Pneumothorax': ['pneumothorax', 'air in pleural', 'apical pneumothorax'],
    'Support Devices': ['tube', 'catheter', 'pacemaker', 'pacer', 'line', 'picc', 'stent']
}

# -------------------------------------------------------------------------
# 2. Run Generation Across All Test Set Instances
# -------------------------------------------------------------------------
print(f"[Evaluation] Generating reports on {len(test_dataset):,} unseen test cases...")

results = []
bleu1_scores, bleu2_scores, bleu3_scores, bleu4_scores = [], [], [], []
rouge1_scores, rouge2_scores, rougeL_scores = [], [], []

gt_pathology_matrix = []
pred_pathology_matrix = []

with torch.no_grad():
    for idx in tqdm(range(len(test_dataset)), desc="Evaluating Test Set"):
        row = test_df.iloc[idx]
        image_tensor = test_dataset[idx]["image"].to(device)
        
        # Greedy autoregressive generation
        gen_tokens = model.greedy_generate(image_tensor, max_len=60, bos_token_id=bos_id, eos_token_id=eos_id)
        pred_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        gt_text = str(row['report']).strip()
        
        # Tokenize words for BLEU calculation
        ref_tokens = [nltk.word_tokenize(gt_text.lower())] if gt_text else [[""]]
        hyp_tokens = nltk.word_tokenize(pred_text.lower()) if pred_text else [""]
        
        # BLEU 1 to 4 with Smoothing
        b1 = sentence_bleu(ref_tokens, hyp_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth)
        b2 = sentence_bleu(ref_tokens, hyp_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
        b3 = sentence_bleu(ref_tokens, hyp_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
        b4 = sentence_bleu(ref_tokens, hyp_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
        
        # ROUGE 1, 2, and L
        rouge_res = rouge_evaluator.score(gt_text, pred_text)
        r1 = rouge_res['rouge1'].fmeasure
        r2 = rouge_res['rouge2'].fmeasure
        rL = rouge_res['rougeL'].fmeasure
        
        bleu1_scores.append(b1)
        bleu2_scores.append(b2)
        bleu3_scores.append(b3)
        bleu4_scores.append(b4)
        
        rouge1_scores.append(r1)
        rouge2_scores.append(r2)
        rougeL_scores.append(rL)
        
        # Binary pathology presence detection
        gt_row_labels = []
        pred_row_labels = []
        for path_name, kws in clinical_keywords.items():
            gt_has = int(any(kw in gt_text.lower() for kw in kws))
            pred_has = int(any(kw in pred_text.lower() for kw in kws))
            gt_row_labels.append(gt_has)
            pred_row_labels.append(pred_has)
            
        gt_pathology_matrix.append(gt_row_labels)
        pred_pathology_matrix.append(pred_row_labels)
        
        results.append({
            "dicom_id": row["dicom_id"],
            "study_id": row["study_id"],
            "ground_truth": gt_text,
            "prediction": pred_text,
            "bleu1": round(b1, 4),
            "bleu4": round(b4, 4),
            "rougeL": round(rL, 4)
        })

# -------------------------------------------------------------------------
# 3. Overall Aggregate Metrics Summary
# -------------------------------------------------------------------------
gt_matrix = np.array(gt_pathology_matrix)
pred_matrix = np.array(pred_pathology_matrix)

micro_f1 = f1_score(gt_matrix, pred_matrix, average='micro', zero_division=0)
macro_f1 = f1_score(gt_matrix, pred_matrix, average='macro', zero_division=0)
micro_prec = precision_score(gt_matrix, pred_matrix, average='micro', zero_division=0)
micro_rec = recall_score(gt_matrix, pred_matrix, average='micro', zero_division=0)

summary_metrics = {
    "Model": "Baseline A (DenseNet121 + Vanilla LSTM)",
    "Test_Samples": len(results),
    "BLEU-1": round(float(np.mean(bleu1_scores)), 4),
    "BLEU-2": round(float(np.mean(bleu2_scores)), 4),
    "BLEU-3": round(float(np.mean(bleu3_scores)), 4),
    "BLEU-4": round(float(np.mean(bleu4_scores)), 4),
    "ROUGE-1": round(float(np.mean(rouge1_scores)), 4),
    "ROUGE-2": round(float(np.mean(rouge2_scores)), 4),
    "ROUGE-L": round(float(np.mean(rougeL_scores)), 4),
    "Clinical_Macro_F1": round(float(macro_f1), 4),
    "Clinical_Micro_F1": round(float(micro_f1), 4),
    "Clinical_Micro_Precision": round(float(micro_prec), 4),
    "Clinical_Micro_Recall": round(float(micro_rec), 4)
}

print("\n" + "=" * 80)
print("OVERALL ACADEMIC BENCHMARK: Baseline A (DenseNet121 + Vanilla LSTM)")
print("=" * 80)
print(f"{'Metric':<30} | {'Value':<15}")
print("-" * 50)
for k, v in summary_metrics.items():
    if k not in ["Model", "Test_Samples"]:
        print(f"{k:<30} | {v:<15}")
print("=" * 80)

# -------------------------------------------------------------------------
# 4. Multilabel Confusion Matrix & Per-Disease Breakdown
# -------------------------------------------------------------------------
pathology_names = list(clinical_keywords.keys())
mcm = multilabel_confusion_matrix(gt_matrix, pred_matrix)
pathology_stats = []

print("\n" + "=" * 95)
print("PER-PATHOLOGY CONFUSION MATRIX & CLINICAL BREAKDOWN")
print("=" * 95)
print(f"{'Pathology':<26} | {'TP':<6} | {'FP':<6} | {'FN':<6} | {'TN':<6} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("-" * 95)

for i, name in enumerate(pathology_names):
    tn, fp, fn, tp = mcm[i].ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    pathology_stats.append({
        "Pathology": name,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Precision": round(float(prec), 4),
        "Recall": round(float(rec), 4),
        "F1_Score": round(float(f1), 4)
    })
    print(f"{name:<26} | {tp:<6} | {fp:<6} | {fn:<6} | {tn:<6} | {prec:<10.4f} | {rec:<10.4f} | {f1:<10.4f}")

print("=" * 95)

# -------------------------------------------------------------------------
# 5. High-Resolution 14-Panel Confusion Matrix Plot
# -------------------------------------------------------------------------
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, name in enumerate(pathology_names):
    cm = mcm[i]
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axes[i],
        xticklabels=["Neg", "Pos"],
        yticklabels=["Neg", "Pos"]
    )
    axes[i].set_title(f"{name}", fontsize=11, fontweight='bold')
    axes[i].set_ylabel("Ground Truth", fontsize=9)
    axes[i].set_xlabel("Predicted", fontsize=9)

for j in range(len(pathology_names), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Multilabel Confusion Matrix: Baseline A (DenseNet121 + LSTM)", fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()

plot_save_path = "/kaggle/working/baseline_a_confusion_matrices.png"
plt.savefig(plot_save_path, dpi=300, bbox_inches='tight')
plt.show()

# -------------------------------------------------------------------------
# 6. Qualitative Case Studies (5 Examples)
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("QUALITATIVE CASE STUDIES: Ground Truth vs Model 0 (LSTM)")
print("=" * 80)
for i in range(min(5, len(results))):
    print(f"\n[Case #{i+1} | Study ID: {results[i]['study_id']}]")
    print(f"  Ground Truth: \"{results[i]['ground_truth']}\"")
    print(f"  Prediction  : \"{results[i]['prediction']}\"")
    print(f"  Scores      : BLEU-4 = {results[i]['bleu4']} | ROUGE-L = {results[i]['rougeL']}")
    print("-" * 80)

# -------------------------------------------------------------------------
# 7. Save All Artifacts to Disk
# -------------------------------------------------------------------------
pd.DataFrame(results).to_csv("/kaggle/working/baseline_a_test_predictions.csv", index=False)
pd.DataFrame(pathology_stats).to_csv("/kaggle/working/baseline_a_pathology_breakdown.csv", index=False)

with open("/kaggle/working/baseline_a_test_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=2)

print("\nEvaluation artifacts saved successfully:")
print("  - /kaggle/working/baseline_a_test_predictions.csv")
print("  - /kaggle/working/baseline_a_pathology_breakdown.csv")
print("  - /kaggle/working/baseline_a_test_metrics.json")
print("  - /kaggle/working/baseline_a_confusion_matrices.png")

[Evaluation] Loading weights from /kaggle/working/baseline_a_best.pt...
[Evaluation] Generating reports on 5,731 unseen test cases...


Evaluating Test Set:   0%|          | 0/5731 [00:00<?, ?it/s]


TypeError: BaselineDenseNetLSTM.greedy_generate() got an unexpected keyword argument 'bos_token_id'